In [1]:
import json

def read_json(filepath):
    return json.load(open(filepath,'r')) 

### Analysis Existing SFT Datas

In [2]:
filepath = '/data/home/zhangjs/disk/project/verl-agent/data_pipelines/gamefiles/sciworld/final/L2_full.json'
sft_data = read_json(filepath=filepath)

filepath1 = '/data/home/zhangjs/disk/project/verl-agent/data_pipelines/gamefiles/sciworld/final/L2_pred_1121.json'
filepath2 = '/data/home/zhangjs/disk/project/verl-agent/data_pipelines/gamefiles/sciworld/final/L2_succ_1121.json'
filepath3 = '/data/home/zhangjs/disk/project/verl-agent/data_pipelines/gamefiles/sciworld/final/L01_train_1080.json'
sft_data = read_json(filepath=filepath1) + read_json(filepath=filepath2) + read_json(filepath=filepath3)

sft_data[0],len(sft_data) 

({'task': 'boil', 'id': 3}, 3322)

In [3]:
win_path1 = '/data/home/zhangjs/disk/project/verl-agent/data_pipelines/datas/sciworld_parallel_coldstart/win_L2_pred.json'
win_path2 = '/data/home/zhangjs/disk/project/verl-agent/data_pipelines/datas/sciworld_parallel_coldstart/win_L2_succ.json'

win = read_json(win_path1) + read_json(win_path2)
win[0]['game_file'],len(win) 

(['measure-melting-point-known-substance', 215], 1326)

### Get the Fail Datas

In [4]:
win_tuple_set = [(elem['game_file'][0],elem['game_file'][1]) for elem in win]
all_tuple_set = [(elem['task'],elem['id']) for elem in sft_data] 

len(set(all_tuple_set)),len(set(win_tuple_set)) 

(3322, 1326)

In [8]:
left = list(set(all_tuple_set) - set(win_tuple_set))
len(left)
left[0] 

('test-conductivity-of-unknown-substances', 115)

In [12]:
import random
list1 = [1,2,3,4]
random.shuffle(list1)
list1

[3, 4, 1, 2]

In [18]:
import random

num_total_samples = 500
total_samples = []

category_dict = {}
for elem in left:
    task = elem[0]
    id = elem[1]
    if task not in category_dict:
        category_dict[task] = [id]
    else:
        category_dict[task].append(id)

for key,value in category_dict.items():
    num_var = len(value)
    ratio = num_var / len(left)
    num_sample = int(ratio * num_total_samples)

    random.shuffle(value)
    for id in value[:num_sample]:
        total_samples.append([key,id])

len(total_samples) 

488

In [19]:
import random
import math

num_total_samples = 500
total_samples = []

category_dict = {}
for elem in left:
    task = elem[0]
    id = elem[1]
    if task not in category_dict:
        category_dict[task] = [id]
    else:
        category_dict[task].append(id)

# 计算每个类别的样本数（使用四舍五入）
category_samples = {}
total_calculated = 0

for key, value in category_dict.items():
    num_var = len(value)
    ratio = num_var / len(left)
    num_sample = round(ratio * num_total_samples)
    category_samples[key] = min(num_sample, len(value))  # 不超过可用样本数
    total_calculated += category_samples[key]

# 调整确保总数正好为500
if total_calculated != num_total_samples:
    # 找到样本最多的类别进行调整
    diff = num_total_samples - total_calculated
    sorted_categories = sorted(category_samples.items(), key=lambda x: x[1], reverse=True)
    
    for i in range(abs(diff)):
        category = sorted_categories[i % len(sorted_categories)][0]
        if diff > 0:
            # 需要增加样本
            if category_samples[category] < len(category_dict[category]):
                category_samples[category] += 1
        else:
            # 需要减少样本
            if category_samples[category] > 0:
                category_samples[category] -= 1

# 执行抽样
for key, value in category_dict.items():
    num_sample = category_samples[key]
    random.shuffle(value)
    for id in value[:num_sample]:
        total_samples.append([key, id])

print(f"最终样本数量: {len(total_samples)}")

最终样本数量: 500


### Prepare Parquert Samples

In [25]:
parquet_train_data = []
for file in total_samples:
    parquet_train_data.append(
        {
            'answer': '',
            'data_source': 'sciworld',
            'prompt': [{'role': 'user', 'content': 'The prompt is dynamic obtained from envs'}],
            'ability': 'agent',
            'gamefile': json.dumps(file),
            'extra_info': {
                'split': 'train'
            }
        }
    )

In [28]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

current_data = parquet_train_data[:500]
df = pd.DataFrame(current_data)

df.to_parquet('/data/home/zhangjs/disk/project/verl-agent/data_pipelines/verl_train_data/sciworld/parallel_train_data_500.parquet', index=False)

In [ ]:
parquet_train_data[0] 

{'answer': '',
 'data_source': 'sciworld',
 'prompt': [{'role': 'user',
   'content': 'The prompt is dynamic obtained from envs'}],
 'ability': 'agent',
 'gamefile': '["test-conductivity-of-unknown-substances", 72]',
 'extra_info': {'split': 'train'}}

In [ ]:
game_files = {}
for idx,elem in enumerate(current_data):
    gamefile = elem['gamefile']
    game_files[str(idx)] = gamefile

with open('../gamefiles/dedu_parallel_train_data_500.json','w') as f:
    json.dump(game_files,f,indent=4)